# Connected to Drive

In [32]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [33]:
path = "/content/drive/My Drive/Revision/history/"

# Import Library

In [34]:
import os
import re
import glob
import itertools
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

# Statistical Result

In [35]:
ALPHA = 0.05
TIE_ATOL = 1e-12

# Proposed ranking metric settings
OPTIMUM_VALUE = 0.0
OPTIMUM_TOL = 1e-8
THRESHOLD_START = 1e3
THRESHOLD_END = 1e-8
N_THRESHOLDS = 51
PROPOSED_W = 0.01

THRESHOLDS = np.logspace(
    np.log10(THRESHOLD_START),
    np.log10(THRESHOLD_END),
    N_THRESHOLDS
)

# ========= YARDIMCI =========
def ensure_dir(p):
    os.makedirs(p, exist_ok=True)
    return p


def to_float_df(df: pd.DataFrame) -> pd.DataFrame:
    # hem 0,123 hem 0.123 desteği
    df = df.copy()

    for c in df.columns:
        s = df[c].astype(str).str.strip().str.replace(",", ".", regex=False)
        df[c] = pd.to_numeric(s, errors="coerce")

    return df


def natural_fun_sort(cols):
    """
    F1..F23 veya 1,2,3 gibi sütunları doğal sıraya dizer.
    """
    def key(c):
        s = str(c).strip()

        m1 = re.match(r"^[Ff](\d+)", s)
        if m1:
            return (0, int(m1.group(1)), s)

        m2 = re.match(r"^(\d+)$", s)
        if m2:
            return (1, int(m2.group(1)), s)

        return (2, 999999, s)

    return sorted(cols, key=key)


def format_p(p: float) -> str:
    if p < ALPHA:
        return f"{p:.4f}*"
    return f"{p:.4f}"


def holm_bonferroni(pvals):
    """
    Holm-Bonferroni adjusted p-value hesaplar.
    """
    pvals = np.array(pvals, dtype=float)
    m = len(pvals)

    if m == 0:
        return np.array([])

    order = np.argsort(pvals)
    p_sorted = pvals[order]

    adj_sorted = (m - np.arange(m)) * p_sorted
    adj_sorted = np.maximum.accumulate(adj_sorted)
    adj_sorted = np.minimum(adj_sorted, 1.0)

    padj = np.empty_like(pvals)
    padj[order] = adj_sorted

    return padj


# ========= VERİ OKUMA =========
def read_model_csv(folder: str, model: str) -> pd.DataFrame:
    path = f"{BASE}/{folder}/best_fit/{model}_best_fit.csv"

    if not os.path.exists(path):
        raise FileNotFoundError(f"Bulunamadı: {path}")

    df = pd.read_csv(path)
    df.columns = [str(c).strip() for c in df.columns]
    df = to_float_df(df)

    return df


def load_all_models(folder: str, models: list) -> dict:
    """
    model -> df(trials x functions)
    """
    out = {}

    for m in models:
        out[m] = read_model_csv(folder, m)

    return out


def common_functions(model2df: dict) -> list:
    """
    Tüm modellerde ortak olan benchmark sütunlarını döndürür.

    Öncelik:
    1) F1, F2, F10 gibi sütunlar
    2) Sadece sayı olan sütunlar: 1, 2, 3...
    3) Yardımcı sütunlar çıkarıldıktan sonra kalan ortak sayısal sütunlar
    """
    inter = None

    for df in model2df.values():
        cols = set(df.columns)
        inter = cols if inter is None else (inter & cols)

    inter = list(inter)

    funcs = [
        c for c in natural_fun_sort(inter)
        if re.match(r"^[Ff]\d+", str(c).strip())
    ]

    if not funcs:
        funcs = [
            c for c in natural_fun_sort(inter)
            if re.match(r"^\d+$", str(c).strip())
        ]

    if not funcs:
        exclude_cols = {
            "run", "runs", "trial", "trials", "seed",
            "iteration", "iterations", "epoch", "epochs",
            "time", "runtime", "elapsed_time",
            "model", "algorithm", "optimizer",
            "solution", "best_solution", "best_fitness",
            "fitness", "score"
        }

        funcs = [
            c for c in natural_fun_sort(inter)
            if str(c).strip().lower() not in exclude_cols
        ]

    if not funcs:
        print("\nOrtak sütunlar:")
        print(inter)
        raise RuntimeError("Modeller arasında ortak benchmark/fonksiyon sütunu bulunamadı.")

    print(f"\nBulunan ortak fonksiyon sayısı: {len(funcs)}")
    print("İlk birkaç fonksiyon sütunu:", funcs[:10])

    return funcs


def trials_matrix_for_function(model2df: dict, func: str, models: list) -> pd.DataFrame:
    """
    Belirli func için n_trials x n_models tablo döndürür.
    Trial sayıları farklıysa minimum trial sayısına kırpar.
    """
    series = []
    used = []
    nmin = None

    for m in models:
        df = model2df[m]

        if func not in df.columns:
            continue

        s = df[func].astype(float).dropna()

        if len(s) == 0:
            continue

        used.append(m)
        nmin = len(s) if nmin is None else min(nmin, len(s))
        series.append(s)

    if not series:
        raise RuntimeError(f"{func} hiçbir modelde bulunamadı veya tüm değerler NaN.")

    series = [s.iloc[:nmin].reset_index(drop=True) for s in series]

    X = pd.concat(series, axis=1)
    X.columns = used

    return X


# ========= PROPOSED CEC-STYLE METRICS =========
def error_values(values, optimum=OPTIMUM_VALUE):
    """
    best_fit değerlerini hata değerine çevirir.
    Dosyada zaten error tutuluyorsa optimum=0 olarak kalmalıdır.
    """
    values = np.asarray(values, dtype=float)
    return np.abs(values - optimum)


def solve_ratio(errors, tol=OPTIMUM_TOL):
    """
    Optimumu bulan run yüzdesi.
    """
    errors = np.asarray(errors, dtype=float)
    errors = errors[~np.isnan(errors)]

    if len(errors) == 0:
        return np.nan

    return np.mean(errors <= tol) * 100


def threshold_ratio(errors, thresholds=THRESHOLDS):
    """
    Geçilen threshold yüzdesi.
    Her run için error <= threshold olan eşik sayısı hesaplanır.
    """
    errors = np.asarray(errors, dtype=float)
    errors = errors[~np.isnan(errors)]

    if len(errors) == 0:
        return np.nan

    passed = [(err <= thresholds).sum() for err in errors]

    return np.sum(passed) / (len(errors) * len(thresholds)) * 100


def _normalize_name(x):
    return str(x).strip().lower().replace(" ", "").replace("-", "").replace("_", "")


def _function_like_columns(df, func):
    """
    Convergence dosyasında fonksiyonla ilişkili sütunları bulur.
    Desteklenen örnekler: F1, f1, F1_run1, run1_F1, F1_trial_3.
    """
    fn = _normalize_name(func)
    cols = []

    for c in df.columns:
        cn = _normalize_name(c)
        if cn == fn or cn.startswith(fn + "run") or cn.startswith(fn + "trial") or cn.endswith("run" + fn) or cn.endswith("trial" + fn):
            cols.append(c)
        elif re.search(rf"(^|[^0-9a-zA-Z]){re.escape(str(func))}([^0-9a-zA-Z]|$)", str(c), flags=re.IGNORECASE):
            cols.append(c)

    return cols


def find_convergence_files(folder: str, model: str, func: str = None):
    """
    Yakınsama eğrilerini şu path yapısında arar:

    BASE/folder/history/convergence/model/F1_best_fit.csv

    Örnek:
    /content/drive/My Drive/Revision/classic/history/convergence/NHO/F1_best_fit.csv
    """

    if func is None:
        return []

    conv_dir = os.path.join(
        BASE,
        folder,
        "history",
        "convergence",
        model
    )

    if not os.path.exists(conv_dir):
        return []

    func = str(func).strip()

    candidates = [
        f"{func}_best_fit.csv",
        f"{func.lower()}_best_fit.csv",
        f"{func.upper()}_best_fit.csv"
    ]

    # F1 -> 1 dönüşümü için
    m = re.match(r"^[Ff](\d+)$", func)

    if m:
        num = int(m.group(1))

        candidates.extend([
            f"{num}_best_fit.csv",
            f"F{num}_best_fit.csv",
            f"f{num}_best_fit.csv",
            f"F{num:02d}_best_fit.csv",
            f"f{num:02d}_best_fit.csv"
        ])

    found = []

    for name in candidates:
        fp = os.path.join(conv_dir, name)

        if os.path.exists(fp):
            found.append(fp)

    # Son çare: wildcard arama
    if not found:
        wildcard_files = glob.glob(os.path.join(conv_dir, "*.csv"))

        for fp in wildcard_files:
            base = os.path.basename(fp).lower()

            if func.lower() in base and "best_fit" in base:
                found.append(fp)

    found = sorted(list(dict.fromkeys(found)))

    return found


def budget_left_ratio_from_curve(curve, max_fes=None, tol=OPTIMUM_TOL):
    """
    Tek convergence curve üzerinden kalan budget yüzdesi.
    Curve değerleri error/best_fit serisi olmalıdır.
    """
    curve = np.asarray(curve, dtype=float)
    curve = curve[~np.isnan(curve)]

    if len(curve) == 0:
        return np.nan

    if max_fes is None:
        max_fes = len(curve)

    if max_fes <= 0:
        return np.nan

    hit_idx = np.where(curve <= tol)[0]

    if len(hit_idx) == 0:
        return 0.0

    first_hit = int(hit_idx[0]) + 1
    return max(0, (max_fes - first_hit) / max_fes) * 100


def budget_left_ratio(folder: str, model: str, func: str):
    """
    Convergence dosyası varsa BudgetLeftRatio hesaplar.
    Dosya yoksa NaN döndürür.

    Not: best_fit dosyaları yalnızca final sonuçları içerdiği için optimumun kaçıncı FES/epoch'ta
    bulunduğu oradan çıkarılamaz. Bu nedenle c3 için convergence/history dosyası gerekir.
    """
    files = find_convergence_files(folder, model, func)

    if not files:
        return np.nan

    ratios = []

    for fp in files:
        try:
            df = pd.read_csv(fp)
            df.columns = [str(c).strip() for c in df.columns]
            df = to_float_df(df)
        except Exception:
            continue

        cols = _function_like_columns(df, func)

        if not cols and len(df.columns) == 1:
            cols = list(df.columns)

        if not cols and func in df.columns:
            cols = [func]

        for c in cols:
            curve = error_values(df[c].astype(float).dropna().values)
            r = budget_left_ratio_from_curve(curve, max_fes=len(curve))
            if not np.isnan(r):
                ratios.append(r)

    if not ratios:
        return np.nan

    return float(np.mean(ratios))


def proposed_metrics_by_function(model2df: dict, funcs: list, models: list, folder: str):
    """
    Her fonksiyon ve model için:
    - SolveRatio
    - ThresholdRatio
    - BudgetLeftRatio
    - Proposed_q
    hesaplar.
    """
    rows = []

    for f in funcs:
        for m in models:
            df = model2df[m]

            if f not in df.columns:
                continue

            vals = df[f].astype(float).dropna().values
            errs = error_values(vals)

            c1 = solve_ratio(errs)
            c2 = threshold_ratio(errs)
            c3 = budget_left_ratio(folder, m, f)

            q = c1 + PROPOSED_W * c2
            if not np.isnan(c3):
                q += (PROPOSED_W ** 2) * c3

            rows.append({
                "Function": f,
                "Model": m,
                "SolveRatio_%": c1,
                "ThresholdRatio_%": c2,
                "BudgetLeftRatio_%": c3,
                "Proposed_q": q,
                "ProposedTriplet": (
                    f"{c1:.2f} / {c2:.2f} / {c3:.2f}"
                    if not np.isnan(c3)
                    else f"{c1:.2f} / {c2:.2f} / NaN"
                )
            })

    metric_df = pd.DataFrame(rows)
    return metric_df


def proposed_metrics_summary(metric_df: pd.DataFrame, models: list):
    """
    Fonksiyonlar üzerinden ortalama metric tablosu.
    """
    summary = (
        metric_df
        .groupby("Model")[["SolveRatio_%", "ThresholdRatio_%", "BudgetLeftRatio_%", "Proposed_q"]]
        .mean()
        .reindex(models)
    )

    summary["ProposedRank"] = summary["Proposed_q"].rank(method="average", ascending=False)
    summary = summary.sort_values("ProposedRank", ascending=True)

    return summary


# ========= HESAPLAR =========
def rank_table_by_mean(model2df: dict, funcs: list, models: list):
    """
    Fonksiyon bazında mean'e göre rank tablosu üretir.
    Küçük mean daha iyi kabul edilir.
    """
    mean_rows = []
    rank_rows = []

    for f in funcs:
        X = trials_matrix_for_function(model2df, f, models)

        means = X.mean(axis=0)
        ranks = means.rank(method="average", ascending=True)

        mean_rows.append({
            "Function": f,
            **{m: means.get(m, np.nan) for m in models}
        })

        rank_rows.append({
            "Function": f,
            **{m: ranks.get(m, np.nan) for m in models}
        })

    mean_df = pd.DataFrame(mean_rows).set_index("Function")
    rank_df = pd.DataFrame(rank_rows).set_index("Function")

    avg_rank = rank_df.mean(axis=0).to_frame().T
    avg_rank.index = ["AverageRank"]

    rank_with_avg = pd.concat([rank_df, avg_rank], axis=0)

    return mean_df, rank_with_avg


def stats_table_by_function(
    model2df: dict,
    funcs: list,
    models: list,
    stats=("min", "mean", "max", "std")
):
    """
    Fonksiyon bazında tüm istatistikleri tek tabloda döndürür.
    Çıktı index: (Function, Statistic)
    Çıktı columns: modeller
    """
    rows = []

    for f in funcs:
        X = trials_matrix_for_function(model2df, f, models)

        if "min" in stats:
            rows.append({
                "Function": f,
                "Statistic": "min",
                **X.min(axis=0).to_dict()
            })

        if "mean" in stats:
            rows.append({
                "Function": f,
                "Statistic": "mean",
                **X.mean(axis=0).to_dict()
            })

        if "max" in stats:
            rows.append({
                "Function": f,
                "Statistic": "max",
                **X.max(axis=0).to_dict()
            })

        if "std" in stats:
            rows.append({
                "Function": f,
                "Statistic": "std",
                **X.std(axis=0, ddof=1).to_dict()
            })

    stats_df = pd.DataFrame(rows).set_index(["Function", "Statistic"])
    stats_df = stats_df.reindex(columns=models)

    return stats_df


def wilcoxon_per_function(model2df: dict, funcs: list, models: list, outdir: str, tag: str):
    """
    Her fonksiyon için tüm model çiftlerinde Wilcoxon + Holm düzeltmesi yapar.
    """
    long_rows = []

    for f in funcs:
        X = trials_matrix_for_function(model2df, f, models)
        algos = list(X.columns)

        pairs = list(itertools.combinations(algos, 2))

        pvals = []
        w_stats = []

        for a, b in pairs:
            x1 = X[a].values
            x2 = X[b].values

            diff = x1 - x2

            if np.all(np.abs(diff) <= TIE_ATOL):
                pvals.append(1.0)
                w_stats.append(0.0)
            else:
                try:
                    res = wilcoxon(
                        x1,
                        x2,
                        zero_method="pratt",
                        alternative="two-sided",
                        mode="auto"
                    )
                    pvals.append(float(res.pvalue))
                    w_stats.append(float(res.statistic))

                except ValueError:
                    pvals.append(1.0)
                    w_stats.append(0.0)

        padj = holm_bonferroni(pvals)

        padj_mat = pd.DataFrame(
            np.ones((len(algos), len(algos))),
            index=algos,
            columns=algos,
            dtype=float
        )

        fmt_mat = pd.DataFrame(
            np.full((len(algos), len(algos)), "—", dtype=object),
            index=algos,
            columns=algos
        )

        for (a, b), p_raw, p_holm, W in zip(pairs, pvals, padj, w_stats):
            padj_mat.loc[a, b] = p_holm
            padj_mat.loc[b, a] = p_holm

            fmt = format_p(p_holm)
            fmt_mat.loc[a, b] = fmt
            fmt_mat.loc[b, a] = fmt

            long_rows.append({
                "Folder": tag,
                "Function": f,
                "Algo_A": a,
                "Algo_B": b,
                "W_stat": W,
                "p_value": p_raw,
                "p_adj_holm": p_holm,
                "Significant": p_holm < ALPHA
            })

        padj_mat.to_csv(os.path.join(outdir, f"{tag}_{f}_padj_matrix.csv"))
        fmt_mat.to_csv(os.path.join(outdir, f"{tag}_{f}_padj_matrix_formatted.csv"))

    wilcoxon_df = pd.DataFrame(long_rows)
    wilcoxon_df.to_csv(
        os.path.join(outdir, f"{tag}_wilcoxon_long_summary.csv"),
        index=False
    )


# ========= ANA =========
def run_folder(folder: str):
    tag = folder

    out_base = ensure_dir(f"{BASE}/{folder}/_stats")
    out_wcx = ensure_dir(f"{out_base}/wilcoxon")

    # 1) yükle
    model2df = load_all_models(folder, MODELS)

    # 2) ortak fonksiyonlar
    funcs = common_functions(model2df)

    # 3) mean, rank ve istatistik tabloları
    mean_df, rank_df = rank_table_by_mean(model2df, funcs, MODELS)

    stats_df = stats_table_by_function(
        model2df,
        funcs,
        MODELS,
        stats=("min", "mean", "max", "std")
    )

    mean_df.to_csv(f"{out_base}/{tag}_mean_by_function.csv")
    stats_df.to_csv(f"{out_base}/{tag}_stats_by_function.csv")
    rank_df.to_csv(f"{out_base}/{tag}_rank_by_mean_per_function.csv")

    # 3B) Proposed CEC-style metrics
    proposed_df = proposed_metrics_by_function(
        model2df=model2df,
        funcs=funcs,
        models=MODELS,
        folder=folder
    )

    proposed_summary_df = proposed_metrics_summary(
        metric_df=proposed_df,
        models=MODELS
    )

    proposed_df.to_csv(
        f"{out_base}/{tag}_proposed_metrics_by_function.csv",
        index=False
    )

    proposed_summary_df.to_csv(
        f"{out_base}/{tag}_proposed_metrics_summary.csv"
    )

    # 4) Wilcoxon + Holm
    wilcoxon_per_function(model2df, funcs, MODELS, out_wcx, tag)

    print(f"\n[{tag}] OK")
    print(f" - stats:            {out_base}/{tag}_stats_by_function.csv")
    print(f" - rank:             {out_base}/{tag}_rank_by_mean_per_function.csv")
    print(f" - proposed summary: {out_base}/{tag}_proposed_metrics_summary.csv")
    print(f" - wcx:              {out_wcx}/{tag}_wilcoxon_long_summary.csv")


    if proposed_summary_df["BudgetLeftRatio_%"].isna().all():
        print(" - note: BudgetLeftRatio_% NaN geldi. Bunun için convergence/history CSV dosyası gerekir.")


In [36]:
# ========= AYAR =========
BASE = path

FOLDERS = ["classic", "CEC22-10D", "CEC22-20D", "CEC20-50D", "CEC20-100D"]

MODELS  = ["NHO", "ACO", "DE", "GA", "GWO", "HGSO", "HHO", "SSO", "ACSA", "BPBO",
           "CHO", "SRA", "L_SHADE", "IMODE", "LSHADEcnEpSin"]

if __name__ == "__main__":
    for folder in FOLDERS:
        run_folder(folder)


Bulunan ortak fonksiyon sayısı: 16
İlk birkaç fonksiyon sütunu: ['Ackley2', 'Alpine2', 'Brent', 'Chichinadze', 'ChungReynolds', 'Cigar', 'CrossInTray', 'Hansen', 'Himmelblau', 'Hosaki']

[classic] OK
 - stats:            /content/drive/My Drive/Revision/history//classic/_stats/classic_stats_by_function.csv
 - rank:             /content/drive/My Drive/Revision/history//classic/_stats/classic_rank_by_mean_per_function.csv
 - proposed summary: /content/drive/My Drive/Revision/history//classic/_stats/classic_proposed_metrics_summary.csv
 - wcx:              /content/drive/My Drive/Revision/history//classic/_stats/wilcoxon/classic_wilcoxon_long_summary.csv
 - note: BudgetLeftRatio_% NaN geldi. Bunun için convergence/history CSV dosyası gerekir.

Bulunan ortak fonksiyon sayısı: 12
İlk birkaç fonksiyon sütunu: ['F12022', 'F22022', 'F32022', 'F42022', 'F52022', 'F62022', 'F72022', 'F82022', 'F92022', 'F102022']

[CEC22-10D] OK
 - stats:            /content/drive/My Drive/Revision/history//CEC

# Summary Results

In [37]:
import os
import numpy as np
import pandas as pd

# =========================================================
# AYARLAR
# =========================================================
BASE = "/content/drive/My Drive/Revision/history"

MODELS  = ["NHO", "ACO", "DE", "GA", "GWO", "HGSO", "HHO", "SSO", "ACSA", "BPBO",
           "CHO", "SRA", "L_SHADE", "IMODE", "LSHADEcnEpSin"]

PROPOSED_MODEL = "NHO"

FOLDERS = ["classic", "CEC22-10D", "CEC22-20D", "CEC20-50D", "CEC20-100D"]


# =========================================================
# YARDIMCI FONKSİYONLAR
# =========================================================
def ensure_dir(p):
    os.makedirs(p, exist_ok=True)
    return p


def sci_fmt(x, precision=3):
    if pd.isna(x):
        return ""
    x = float(x)
    if x == 0:
        return "0.000"
    if abs(x) < 1e-3 or abs(x) >= 1e4:
        return f"{x:.{precision}E}"
    return f"{x:.3f}"


def read_stats_files(base, folder):
    stat_dir = f"{base}/{folder}/_stats"

    files = {
        "stats": f"{stat_dir}/{folder}_stats_by_function.csv",
        "rank": f"{stat_dir}/{folder}_rank_by_mean_per_function.csv",
        "wilcoxon": f"{stat_dir}/wilcoxon/{folder}_wilcoxon_long_summary.csv",
        "proposed": f"{stat_dir}/{folder}_proposed_metrics_summary.csv"
    }

    for k, fp in files.items():
        if not os.path.exists(fp):
            raise FileNotFoundError(f"{k} dosyası bulunamadı: {fp}")

    stats_df = pd.read_csv(files["stats"])
    rank_df = pd.read_csv(files["rank"])
    wilcoxon_df = pd.read_csv(files["wilcoxon"])
    proposed_df = pd.read_csv(files["proposed"])

    return stats_df, rank_df, wilcoxon_df, proposed_df


# =========================================================
# TABLO 1: NHO İSTATİSTİK TABLOSU
# =========================================================
def create_nho_statistical_table(stats_df, proposed_model="NHO"):
    """
    stats_by_function.csv dosyasından yalnızca önerilen modelin
    Min, Mean, Max, Std değerlerini üretir.
    """

    df = stats_df.copy()

    if "Function" not in df.columns or "Statistic" not in df.columns:
        df = df.rename(columns={df.columns[0]: "Function", df.columns[1]: "Statistic"})

    needed_stats = ["min", "mean", "max", "std"]

    rows = []

    for func in df["Function"].dropna().unique():
        sub = df[df["Function"] == func]

        row = {"Function": func}

        for st in needed_stats:
            val = sub.loc[sub["Statistic"].astype(str).str.lower() == st, proposed_model]
            row[st.capitalize() if st != "std" else "Std"] = val.iloc[0] if len(val) > 0 else np.nan

        rows.append(row)

    out = pd.DataFrame(rows)

    for c in ["Min", "Mean", "Max", "Std"]:
        out[c] = out[c].apply(sci_fmt)

    return out


# =========================================================
# TABLO 2: AVERAGE FRIEDMAN RANKING
# =========================================================
def create_average_rank_table(rank_df):
    """
    rank_by_mean_per_function.csv dosyasından AverageRank satırını okur.
    """

    df = rank_df.copy()
    first_col = df.columns[0]

    avg_row = df[df[first_col].astype(str) == "AverageRank"]

    if avg_row.empty:
        raise ValueError("AverageRank satırı bulunamadı.")

    avg_row = avg_row.iloc[0].drop(labels=[first_col])

    out = pd.DataFrame({
        "Algorithm": avg_row.index,
        "Average Rank": avg_row.values.astype(float)
    })

    out = out.sort_values("Average Rank", ascending=True).reset_index(drop=True)
    out["Average Rank"] = out["Average Rank"].map(lambda x: f"{x:.3f}")

    return out


# =========================================================
# TABLO 3: NHO WINS-TIES-LOSSES
# =========================================================
def create_wilcoxon_wtl_table(wilcoxon_df, proposed_model="NHO", alpha=0.05):
    """
    wilcoxon_long_summary.csv dosyasından NHO vs diğer algoritmalar için
    Wins-Ties-Losses üretir.

    Mantık:
    - p_adj_holm >= alpha ise Tie
    - anlamlı fark varsa ortalama/median yönü olmadığı için burada yalnızca
      Significance bilgisi yetmez.
    - Bu yüzden WTL için ayrıca Algo_A/Algo_B yönüne göre değil,
      sonuç dosyasındaki p değerlerinden değil mean rank dosyasından destek almak gerekir.

    Bu fonksiyon p-value tabanlı tie sayısını verir.
    Win/Loss ayrımı için stats_df de gerekir.
    """

    raise NotImplementedError(
        "WTL için yön bilgisi gerekir. create_wilcoxon_wtl_table_with_stats() kullanın."
    )


def create_wilcoxon_wtl_table_with_stats(
    wilcoxon_df,
    stats_df,
    proposed_model="NHO",
    alpha=0.05
):
    """
    Holm düzeltilmiş Wilcoxon sonucunu stats_by_function mean değerleriyle birleştirir.
    Küçük mean daha iyi kabul edilir.
    """

    means = stats_df.copy()

    if "Function" not in means.columns or "Statistic" not in means.columns:
        means = means.rename(columns={means.columns[0]: "Function", means.columns[1]: "Statistic"})

    means = means[means["Statistic"].astype(str).str.lower() == "mean"]

    algorithms = sorted(
        set(wilcoxon_df["Algo_A"]).union(set(wilcoxon_df["Algo_B"])) - {proposed_model}
    )

    rows = []

    for alg in algorithms:
        wins, ties, losses = 0, 0, 0

        pair_df = wilcoxon_df[
            ((wilcoxon_df["Algo_A"] == proposed_model) & (wilcoxon_df["Algo_B"] == alg)) |
            ((wilcoxon_df["Algo_A"] == alg) & (wilcoxon_df["Algo_B"] == proposed_model))
        ]

        for _, r in pair_df.iterrows():
            func = r["Function"]
            p_adj = r["p_adj_holm"]

            if p_adj >= alpha:
                ties += 1
                continue

            mean_row = means[means["Function"] == func]

            if mean_row.empty:
                continue

            nho_mean = float(mean_row[proposed_model].iloc[0])
            alg_mean = float(mean_row[alg].iloc[0])

            if nho_mean < alg_mean:
                wins += 1
            elif nho_mean > alg_mean:
                losses += 1
            else:
                ties += 1

        rows.append({
            "Algorithm": alg,
            "Wins": wins,
            "Ties": ties,
            "Losses": losses
        })

    out = pd.DataFrame(rows)
    out = out.sort_values(["Wins", "Ties", "Losses"], ascending=[False, False, True])
    out = out.reset_index(drop=True)

    return out


# =========================================================
# TABLO 4: ALTERNATIVE / PROPOSED METRICS
# =========================================================
def create_proposed_metrics_table(proposed_df, top_n=5):
    """
    proposed_metrics_summary.csv dosyasından alternatif metrik tablosunu üretir.
    """

    df = proposed_df.copy()

    if "Model" not in df.columns:
        df = df.rename(columns={df.columns[0]: "Model"})

    out = df[["Model", "SolveRatio_%", "ThresholdRatio_%", "Proposed_q", "ProposedRank"]].copy()

    out = out.rename(columns={
        "Model": "Algorithm",
        "SolveRatio_%": "Solve Ratio (%)",
        "ThresholdRatio_%": "Threshold Ratio (%)",
        "Proposed_q": "Proposed q",
        "ProposedRank": "Rank"
    })

    out = out.sort_values("Rank", ascending=True)

    if top_n is not None:
        out = out.head(top_n)

    out["Solve Ratio (%)"] = out["Solve Ratio (%)"].map(lambda x: f"{x:.2f}")
    out["Threshold Ratio (%)"] = out["Threshold Ratio (%)"].map(lambda x: f"{x:.3f}")
    out["Proposed q"] = out["Proposed q"].map(lambda x: f"{x:.3f}")
    out["Rank"] = out["Rank"].map(lambda x: int(x) if float(x).is_integer() else x)

    return out.reset_index(drop=True)


# =========================================================
# TEK FOLDER İÇİN 4 TABLOYU ÜRET
# =========================================================
def create_all_manuscript_tables_for_folder(
    base,
    folder,
    proposed_model="NHO",
    top_n_proposed=5,
    save=True
):
    stats_df, rank_df, wilcoxon_df, proposed_df = read_stats_files(base, folder)

    table_stats = create_nho_statistical_table(
        stats_df,
        proposed_model=proposed_model
    )

    table_rank = create_average_rank_table(rank_df)

    table_wtl = create_wilcoxon_wtl_table_with_stats(
        wilcoxon_df,
        stats_df,
        proposed_model=proposed_model
    )

    table_metrics = create_proposed_metrics_table(
        proposed_df,
        top_n=top_n_proposed
    )

    if save:
        out_dir = ensure_dir(f"{base}/{folder}/_manuscript_tables")

        table_stats.to_csv(f"{out_dir}/{folder}_Table_statistical_results_{proposed_model}.csv", index=False)
        table_rank.to_csv(f"{out_dir}/{folder}_Table_average_friedman_rank.csv", index=False)
        table_wtl.to_csv(f"{out_dir}/{folder}_Table_wilcoxon_WTL_{proposed_model}.csv", index=False)
        table_metrics.to_csv(f"{out_dir}/{folder}_Table_alternative_metrics.csv", index=False)

        print(f"\n[{folder}] tablolar kaydedildi:")
        print(f" - {out_dir}/{folder}_Table_statistical_results_{proposed_model}.csv")
        print(f" - {out_dir}/{folder}_Table_average_friedman_rank.csv")
        print(f" - {out_dir}/{folder}_Table_wilcoxon_WTL_{proposed_model}.csv")
        print(f" - {out_dir}/{folder}_Table_alternative_metrics.csv")

    return table_stats, table_rank, table_wtl, table_metrics


# =========================================================
# TÜM BENCHMARKLAR İÇİN ÇALIŞTIR
# =========================================================
def create_all_manuscript_tables(
    base,
    folders,
    proposed_model="NHO",
    top_n_proposed=5
):
    results = {}

    for folder in folders:
        results[folder] = create_all_manuscript_tables_for_folder(
            base=base,
            folder=folder,
            proposed_model=proposed_model,
            top_n_proposed=top_n_proposed,
            save=True
        )

    return results


# =========================================================
# ÇALIŞTIRMA
# =========================================================
all_tables = create_all_manuscript_tables(
    base=BASE,
    folders=FOLDERS,
    proposed_model=PROPOSED_MODEL,
    top_n_proposed=5
)


[classic] tablolar kaydedildi:
 - /content/drive/My Drive/Revision/history/classic/_manuscript_tables/classic_Table_statistical_results_NHO.csv
 - /content/drive/My Drive/Revision/history/classic/_manuscript_tables/classic_Table_average_friedman_rank.csv
 - /content/drive/My Drive/Revision/history/classic/_manuscript_tables/classic_Table_wilcoxon_WTL_NHO.csv
 - /content/drive/My Drive/Revision/history/classic/_manuscript_tables/classic_Table_alternative_metrics.csv

[CEC22-10D] tablolar kaydedildi:
 - /content/drive/My Drive/Revision/history/CEC22-10D/_manuscript_tables/CEC22-10D_Table_statistical_results_NHO.csv
 - /content/drive/My Drive/Revision/history/CEC22-10D/_manuscript_tables/CEC22-10D_Table_average_friedman_rank.csv
 - /content/drive/My Drive/Revision/history/CEC22-10D/_manuscript_tables/CEC22-10D_Table_wilcoxon_WTL_NHO.csv
 - /content/drive/My Drive/Revision/history/CEC22-10D/_manuscript_tables/CEC22-10D_Table_alternative_metrics.csv

[CEC22-20D] tablolar kaydedildi:
 - /co